In [26]:
import pandas as pd

In [ ]:
raw_data = pd.read_csv(r"./test_files/balances_report_556_49.csv",
                      dtype={
                          "contact_number": "string"
                      })

raw_data

In [ ]:
clean_data = raw_data.loc[:, :]

clean_data["new_balance"] = clean_data["cleared_balance"] - clean_data["points_earned"]

clean_data = clean_data.loc[:, ["contact_number", "new_balance"]]

valid_contact_mask = clean_data["contact_number"].astype(str).str.len() == 12

discarded_data =clean_data[
    (clean_data["new_balance"] <= 0) | ~valid_contact_mask
]

clean_data = clean_data[
    (clean_data["new_balance"] > 0) & valid_contact_mask
]

discarded_data.to_csv(
    r"./test_files/points_to_expire_2_columns_discarded.csv",
    index=False,
    header=False,
)
clean_data.to_csv(
    r"./test_files/points_to_expire_2_columns.csv",
    index=False,
    header=False,
)

clean_data

In [29]:
total_rows = raw_data.shape[0]
kept_rows = clean_data.shape[0]
discarded_rows = discarded_data.shape[0]

assert kept_rows + discarded_rows == total_rows, "Row count mismatch: some rows lost or duplicated"
print(
    f"Kept={kept_rows}, Discarded={discarded_rows}, Total={total_rows}"
)

Kept=711443, Discarded=98030, Total=809473


In [ ]:
chunksize = 100_000
for i, chunk in enumerate(pd.read_csv(r"./test_files/points_to_expire_2_columns.csv", chunksize=chunksize)):
    print(
        f"Writing part {i + 1}, rows={len(chunk)}"
    )  # will show smaller size for last chunk
    chunk.to_csv(f"./test_files/points_to_expire_{i + 1:03d}.csv", index=False, header=False)